# 04 — Parquet, write modes and partitioned output

**Student exercise**

Complete the tasks in order. Each task includes an expected result, three hints, and a check. All sample data and setup code are included in this notebook.

## 1. Spark configuration

Start a fresh kernel. Use Spark 3.5.x / Scala 2.12 and Java 17. Set `STUDENT_ID` to a lowercase name containing letters, numbers or underscores. Set `SPARK_MASTER=local[2]` for a local Spark session.

SeaweedFS S3 endpoint: `http://localhost:9001`. Use an existing `datalake` bucket. Setup and write tasks overwrite only the specified exercise prefix; use a distinct student ID. The driver and workers must be able to reach this endpoint. Hadoop AWS 3.3.4 must match the Hadoop version in Spark; connector JARs are downloaded from Maven.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
STUDENT_ID = "student01"
assert STUDENT_ID and STUDENT_ID[0].islower() and all(c in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in STUDENT_ID)
base_path = f"s3a://datalake/exercises/{STUDENT_ID}/ex04"

spark = (
    SparkSession.builder
    .appName("Exercise-04")
    .master(master_url)
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9001")
    .config("spark.hadoop.fs.s3a.access.key", "team")
    .config("spark.hadoop.fs.s3a.secret.key", "team1234")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Sample data

Run this cell, inspect the records, and use the supplied variables in the tasks.

In [ ]:
orders = spark.createDataFrame([
    (101,1,40.0,'2026-09-01'), (102,2,50.0,'2026-09-01'),
    (103,1,30.0,'2026-09-02'), (104,3,20.0,'2026-09-02')],
    'id INT, customer_id INT, amount DOUBLE, order_date STRING')
orders.show()

## 3. Result checks

This small function checks output values and duplicate counts. It is included here so the notebook can run on its own.

In [ ]:
def check_frame(df, columns, expected):
    assert df is not None, "Complete the task before running the check"
    assert set(columns).issubset(df.columns), "Missing output columns"
    def normalize(row):
        return tuple(round(value, 6) if isinstance(value, float) else value for value in row)
    actual = sorted(repr(normalize(row)) for row in df.select(*columns).collect())
    wanted = sorted(repr(normalize(row)) for row in expected)
    assert actual == wanted, f"Expected {wanted}, got {actual}"
    print("PASS:", columns)

## Task 1: Partition a Parquet dataset

Write orders under f"{base_path}/output/partitioned" partitioned by order_date and read it as partitioned.

**Expected result:** Four rows and a discovered date partition column.

In [ ]:
partitioned = None  # TODO: write.partitionBy and read.parquet

**Hint 1:** Partition values can be inferred from folder names when reading the root.

**Hint 2:** partitionBy belongs to DataFrameWriter, not DataFrameReader.

**Hint 3:** orders.write.mode(...).partitionBy("order_date").parquet(uri).

### Check

In [ ]:
check_frame(partitioned,['id','amount'],[(101,40.0),(102,50.0),(103,30.0),(104,20.0)])
assert 'order_date' in partitioned.columns

## Task 2: Observe append versus overwrite

At a separate modes prefix: write orders once, append id=101 once, observe five rows, then overwrite with orders and observe four.

**Expected result:** Append gives five rows; overwrite restores four.

In [ ]:
appended = None
reset = None  # TODO: separate prefix and read after each mode

**Hint 1:** Appending files does not enforce unique IDs.

**Hint 2:** Materialize the appended result before overwriting its backing path; Spark is lazy.

**Hint 3:** Use cache()+count() before the overwrite, or collect the before-state into a small new DataFrame.

### Check

In [ ]:
check_frame(appended,['id'],[(101,),(101,),(102,),(103,),(104,)])
check_frame(reset,['id'],[(101,),(102,),(103,),(104,)])
appended.unpersist()

## Task 3: Observe ignore and error-if-exists

Write with ignore to the existing modes prefix and retain it as ignored. Catch the expected default/error-if-exists exception; set error_observed=True only on failure.

**Expected result:** Existing output unchanged; expected conflict observed.

In [ ]:
ignored = None
error_observed = False  # TODO: handle the expected write exception

**Hint 1:** Ignore preserves an existing output; it is not an append mode.

**Hint 2:** Only catch the expected existing-path error; let network/permission failures surface.

**Hint 3:** try write.mode("errorifexists"); except: inspect the exception before marking it expected.

### Check

In [ ]:
check_frame(ignored,['id'],[(101,),(102,),(103,),(104,)])
assert error_observed, 'Expected an existing-path error'

## More practice

Compare snappy and gzip Parquet files using schemas and row counts; do not infer performance from four rows.

## Common mistakes

A lazy DataFrame read from a path can become invalid when that path is overwritten. Append does not deduplicate.

## Finish

Stop Spark before starting another notebook. For write exercises, run setup again before repeating append tasks. S3 data remains available after stopping Spark.

In [ ]:
spark.stop()